In [0]:
from pyspark.sql import functions as F
import builtins

# Candidate pairs saved by the duplicate detection notebook (04)
DATABASE_NAME = "smart_data_deduplication"
candidate_pairs = spark.table(
    f"{DATABASE_NAME}.duplicate_detection_results"
)

# Total candidate pairs
total_candidates = candidate_pairs.count()

# Thresholds for evaluation
thresholds = [0.70, 0.80, 0.90, 0.93]

results = []

for threshold in thresholds:

    accepted = candidate_pairs.filter(
        F.col("similarity_score") >= threshold
    ).count()

    rejected = total_candidates - accepted

    match_rate = (
        builtins.round(
            accepted * 100.0 / total_candidates, 2
        ) if total_candidates > 0 else 0.0
    )

    label = (
        "Selected threshold (0.93)"
        if threshold == 0.93
        else f"{threshold:.2f}"
    )

    results.append((
        label,
        total_candidates,
        accepted,
        rejected,
        match_rate
    ))

# Create Table 5
table5_df = spark.createDataFrame(
    results,
    [
        "Similarity Threshold",
        "Candidate Pairs",
        "Accepted Matches",
        "Rejected Matches",
        "Match Rate"
    ]
)

table5_df = table5_df.withColumn(
    "Match Rate",
    F.concat(
        F.format_number(F.col("Match Rate"), 2),
        F.lit("%")
    )
)

display(table5_df)

# Save to Delta (column names cannot contain spaces)
table5_df.toDF(
    "similarity_threshold",
    "candidate_pairs",
    "accepted_matches",
    "rejected_matches",
    "match_rate"
).write.format("delta").mode("overwrite").saveAsTable(
    f"{DATABASE_NAME}.similarity_threshold_evaluation"
)

Similarity Threshold,Candidate Pairs,Accepted Matches,Rejected Matches,Match Rate
0.70,2,2,0,100.00%
0.80,2,2,0,100.00%
0.90,2,2,0,100.00%
Selected threshold (0.93),2,2,0,100.00%
